## Requirements

Place these files in the SAME directory:
```bash
main.py
build_vectors_13b_chat.ipynb
```

Export HF token before launching (Llama-2-13b-chat-hf is a **gated model** — token is mandatory):
```bash
export HF_TOKEN="your_token"
```

**GPU requirement:** Llama-2-13b in bfloat16 needs ~26 GB VRAM. Use an **A100 (40 GB)** runtime in Colab.


In [ ]:
# Cell 1 — Install dependencies

!pip install -q transformers==4.40.0 torch scikit-learn accelerate huggingface_hub

print("✓ Dependencies installed")


In [ ]:
# Cell 2 — Authenticate with Hugging Face
# Llama-2-13b-chat-hf is a gated model — HF_TOKEN is mandatory.

import os
from huggingface_hub import login

HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN not found. Run:\n"
        "export HF_TOKEN='your_token'\n"
        "You must also have accepted the Llama-2 licence at:\n"
        "https://huggingface.co/meta-llama/Llama-2-13b-chat-hf"
    )

login(token=HF_TOKEN)

print("✓ Hugging Face login successful")


In [ ]:
# Cell 3 — Verify GPU + versions
import torch, transformers
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM         : {vram_gb:.1f} GB')
    if vram_gb < 26.0:
        print('WARNING: Llama-2-13b-chat-hf needs ~26 GB VRAM in bfloat16. Upgrade to A100 (40 GB).')
    else:
        print('✓ Sufficient VRAM for Llama-2-13b-chat-hf in bfloat16')
else:
    print('WARNING: No GPU — this will be extremely slow. Runtime → Change runtime type → A100 GPU')

In [ ]:
# Cell 4 — Verify main.py exists

from pathlib import Path

main_file = Path("main.py")

if not main_file.exists():
    raise FileNotFoundError(
        "main.py not found in current directory."
    )

print(f"✓ Found main.py at: {main_file.resolve()}")


In [ ]:
# Cell 5 — Configure environment and import from main.py
#
# Key differences from Llama-2-7b:
#   LOCAL_MODEL_NAME : Llama-2-13b-chat-hf  (instruction-tuned, gated)
#   STEER_LAYER      : 28   (13b has 40 layers; 28 ≈ 70% depth, same relative position as layer 20 in 7b)
#   STEER_ALPHA      : 12.0 (13b hidden_dim=5120 vs 4096; lower alpha avoids over-steering)

import os
import importlib.util
from pathlib import Path

os.environ["LOCAL_MODEL_NAME"] = "meta-llama/Llama-2-13b-chat-hf"
os.environ["STEER_LAYER"]      = "28"
os.environ["STEER_ALPHA"]      = "12.0"
os.environ["STYLE_CACHE_DIR"]  = ".style_cache"
os.environ["HF_TOKEN"]         = HF_TOKEN   # forward token so main.py model loader picks it up

Path(".style_cache").mkdir(exist_ok=True)

spec = importlib.util.spec_from_file_location("main_module", "main.py")
main_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(main_module)

print("✓ main.py imported successfully")
print("=" * 60)
print("Model      :", os.environ["LOCAL_MODEL_NAME"])
print("Steer layer:", os.environ["STEER_LAYER"])
print("Steer alpha:", os.environ["STEER_ALPHA"])
print("Cache dir  :", os.environ["STYLE_CACHE_DIR"])
print("=" * 60)


In [ ]:
# Cell 6 — Build EMPATHETIC vector

print("Building EMPATHETIC vector (this loads Llama-2-13b-chat-hf — expect ~3-5 min on A100)...")

vec_emp = main_module.build_style_vector(
    style="empathetic",
    method="pca"
)

print("\n✓ EMPATHETIC vector built successfully")


In [ ]:
# Cell 7 — Build FORMAL vector
# Model is already cached in memory from Cell 6 — this cell is much faster.

print("Building FORMAL vector (model already loaded — expect ~1-2 min on A100)...")

vec_formal = main_module.build_style_vector(
    style="formal",
    method="pca"
)

print("\n✓ FORMAL vector built successfully")


In [ ]:
# Cell 8 — Verify vectors are valid
import torch

# Use consistent variable name (vec_formal, not vec_form)
vec_emp  = vec_emp
vec_form = vec_formal

sim = torch.nn.functional.cosine_similarity(
    vec_emp.unsqueeze(0), vec_form.unsqueeze(0)
).item()

HIDDEN_DIM = 5120   # Llama-2-13b hidden dimension (was 4096 for 7b)

print('='*55)
print(f'  empathetic : norm={vec_emp.norm():.6f}  shape={list(vec_emp.shape)}')
print(f'  formal     : norm={vec_form.norm():.6f}  shape={list(vec_form.shape)}')
print(f'  cosine sim : {sim:.4f}')
print()

assert abs(vec_emp.norm().item()  - 1.0) < 0.01, 'empathetic vector not unit-norm!'
assert abs(vec_form.norm().item() - 1.0) < 0.01, 'formal vector not unit-norm!'
assert vec_emp.shape[0]  == HIDDEN_DIM, f'Expected hidden_dim={HIDDEN_DIM} for Llama-2-13b, got {vec_emp.shape[0]}'
assert vec_form.shape[0] == HIDDEN_DIM, f'Expected hidden_dim={HIDDEN_DIM} for Llama-2-13b, got {vec_form.shape[0]}'
assert sim < 0.5, f'Cosine similarity {sim:.3f} too high — vectors may not be distinct!'

if sim < 0:
    print('✓ Cosine similarity is NEGATIVE — vectors point in opposite directions (expected)')
else:
    print(f'  Cosine similarity {sim:.3f} > 0 — vectors diverge but are not opposite.')
    print('  This is acceptable — the important thing is they are distinct directions.')

import os
cache_files = list(os.listdir('.style_cache'))
print(f'\nCached files in .style_cache/:')
for f in cache_files:
    size = os.path.getsize(f'.style_cache/{f}') / 1024
    print(f'  {f}  ({size:.1f} KB)')
print('\n✓ All checks passed!')
print(f'  Note: Each pkl is ~20 KB (5120 float32 values) — larger than 7b\'s 16 KB (4096 floats)')

In [ ]:
# Cell 9 — Quick smoke test: does the hook actually steer generation?
#
# Changes from 7b version:
#   - Prompt uses full Llama-2-chat <<SYS>> format (matching main.py)
#   - ALPHA_TEST = 12.0  (was 15.0 for 7b; 13b is more sensitive due to larger hidden dim)
#   - Hook steers ALL positions, not just [:, -1, :] (matches the fix in main.py)

import torch

ALPHA_TEST = 12.0

# Use the full chat format that Llama-2-13b-chat-hf was trained on
test_prompt = (
    '<s>[INST] <<SYS>>\n'
    'You are a professional customer support agent. Write a support reply. '
    'Do NOT use headers or bullet points. Write ONLY the reply — 3 to 4 sentences maximum.\n'
    '<</SYS>>\n\n'
    'Customer message: My <PRODUCT> order <ORDER_ID> has an <ISSUE>. [/INST] '
)

model, tokenizer = main_module._get_model_and_tokenizer()

def generate_steered(prompt, style_vec, alpha, label):
    device = next(model.parameters()).device
    vec    = style_vec.to(device=device, dtype=torch.float32)
    layer  = main_module._get_layer(model)

    def _hook(module, inp, output):
        h = output[0] if isinstance(output, tuple) else output
        h_f32 = h.float()
        # Steer ALL positions (matches fix in main.py — not just [:, -1, :])
        h_f32 = h_f32 + alpha * vec
        steered = h_f32.to(h.dtype)
        return (steered,) + output[1:] if isinstance(output, tuple) else steered

    handle = layer.register_forward_hook(_hook)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    try:
        with torch.no_grad():
            out = model.generate(
                inputs.input_ids,
                max_new_tokens=80,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.3,
            )
    finally:
        handle.remove()

    new_ids = out[0][inputs.input_ids.shape[1]:]
    text    = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    print(f'\n[{label}]  {text[:300]}')
    return text

print('=== Smoke Test: Same prompt, two style vectors ===')
print(f'Prompt: {test_prompt}')

r_emp  = generate_steered(test_prompt, vec_emp,  ALPHA_TEST, 'EMPATHETIC')
r_form = generate_steered(test_prompt, vec_form, ALPHA_TEST, 'FORMAL')

print('\n=== Smoke test complete — check that the two responses have different tones ===')

In [ ]:
# Cell 10 — Verify saved vectors

from pathlib import Path

cache_dir = Path(".style_cache")

print("=" * 60)
print("Stored vectors")
print("=" * 60)

for file in sorted(cache_dir.glob("*")):
    size_kb = file.stat().st_size / 1024
    print(f"{file.name:<45} {size_kb:.2f} KB")

print("=" * 60)
print("Absolute cache path:")
print(cache_dir.resolve())
print()
print("Copy the .style_cache/ folder next to main.py before running the pipeline.")
